# Continuous Integration

<div class="alert alert-info">

### Aims:
 - Become familiar with the basic concepts of continuous integration (CI)
 - Gain basic knowledge of various existing CI frameworks
 - Become familiar with the basic use of **GitHub Actions**, including:
    - Setting up basic testing workflows for a Python package
    - Utilizing actions within a workflow
    - Knowing where workflows run (GitHub's runners: virtual machines in the cloud) and what that means for them



<div class="alert alert-info">

### Notes

- This is the second half of today's lecture. It starts from a small package laid out like the one in the first half (`python-packaging.ipynb`): a `pyproject.toml`, a package folder and a `tests` folder, with tests written for `pytest` (Lecture 4).
- The package is in the `simple_functions` folder within `Lecture05`. You'll give a copy of it a GitHub repository of its own.
- Everything you'll run on your own machine (`pytest`, `flake8` and `numpy`) is already in the module's `mpm2026` environment. Run the commands in your Ubuntu (WSL) terminal.

</div>

## What is continuous integration?

In software development, Continuous Integration (CI) is the practice of developers integrating code into a shared repository several times a day. Each 'push' to the repository is then verified by automated testing tools, allowing any problems to be detected early.

And, from [Wikipedia](https://en.wikipedia.org/wiki/Continuous_integration) (October 2026):

_Continuous integration (CI) is the practice of integrating source code changes frequently and ensuring that the integrated codebase is in a workable state. Typically, developers merge changes to an integration branch, and an automated system builds and tests the software system. Often, the automated process runs on each commit or runs on a schedule such as once a day. Grady Booch first proposed the term CI in 1991, although he did not advocate integrating multiple times a day, but later, CI came to include that aspect._

## Automated continuous integration

Once you have tests, you want to make sure that any public changes to the code don't break them. For code held under version control, this means that you ideally want to rerun your test suite on all new public merges, to ensure that the code installs and runs successfully on a clean system. Continuous integration (CI) and Continuous Delivery (CD) frameworks tie together version control repositories and test suites so that this happens automatically whenever a commit to production code (i.e. your _main_ branch) is proposed, or even whenever any change is committed anywhere in the repository.

The goal is to first ensure that the production branch always works, so that users can trust they will always be able to use it securely. Meanwhile, developers can feel safe in repeatedly merging changes to and from the main version and knowing whether their code works or not. At its furthest, under the doctrine of Extreme Programming (XP), this might mean work can be merged multiple times a day by every developer among moderately sized teams.

XP workflows stress the following points:

 - Merge back to main frequently (i.e. keep feature changes small and atomic)
 - Automate the software build
 - Automate the testing framework

## Some CI solutions

As with test frameworks, a large number of CI/CD solutions exist, e.g.

 - [GitLab CI/CD](https://docs.gitlab.com/ci/), built into GitLab (an alternative to GitHub, see Lecture 1).
 - [Jenkins](https://www.jenkins.io), a free, open source CI tool which you run on your own machines. Popular with Java developers.
 - [Buildbot](https://buildbot.net) - Another self-run CI tool, written and configured with Python.
 - [Azure Pipelines](https://azure.microsoft.com/en-gb/products/devops/pipelines/) - A hosted (cloud) CI service, run by Microsoft.
 - [Travis CI](https://www.travis-ci.com/), the hosted service that ran the tests of most open-source projects on GitHub until it stopped being free for them at the end of 2020. You'll still see `.travis.yml` files in older repositories.

and of course, the main CI pipeline we will learn about today

 - [GitHub Actions](https://docs.github.com/en/actions), which allows building continuous integration and continuous deployment pipelines for testing, releasing and deploying software without the use of third-party websites/platforms

All the above CI solutions require their own setup files (e.g. `.gitlab-ci.yml` for GitLab, `azure-pipelines.yml` for Azure Pipelines), which have many of the same concepts (when to run, on what kind of machine, and which commands), but different syntax.

In general, which CI framework to use is rarely a true choice. Businesses and existing projects will have picked and offer support for one, and expect you to play nicely with it. Meanwhile for personal projects people want minimum effort, minimum cost solutions. On these two metrics GitHub Actions scores very highly.

GitHub Actions was added to GitHub in 2019. It is still developing quickly: new features are added and existing ones changed regularly, so check the documentation, and the versions of the actions you use, from time to time.

## GitHub Actions

To begin familiarising ourselves with Actions, let us dive straight into an example: the `simple_functions` package in the `Lecture05/simple_functions` folder. It is laid out just like the package in the first half of today's lecture:

```text
simple_functions/
 ├── pyproject.toml
 ├── LICENSE
 ├── README.md
 ├── .gitignore
 ├── simple_functions/
 │    ├── __init__.py
 │    └── functions1.py
 └── tests/
      └── test_simple_functions.py
```

Start by giving it a repository of its own on GitHub, as in Lecture 1:

1. On GitHub, create a new repository called `simple_functions` in your personal account (tick *Add README*: it is replaced by the package's own in a moment). It can be public or private; see the note on minutes below.
2. Clone it, somewhere outside your clone of the course repository, and copy the package into it (the `/.` at the end of the `cp` source copies hidden files such as `.gitignore` too):

```bash
git clone https://github.com/<my username>/simple_functions.git
cp -r <path to>/modern-programming-methods/Lecture05/simple_functions/. simple_functions/
cd simple_functions
git add .
git commit -m "Add the simple_functions package"
git push
```

Now, let's install the package (in editable mode, with its `dev` extras, as in the first half) and check, on your own machine first, that its tests pass and that `flake8` is happy:

```bash
conda activate mpm2026
pip install -e ".[dev]"
pytest
flake8 .
```
`pytest` should report `2 passed`, and `flake8` should print nothing. (`pip show simple_functions` shows where the package is installed; `pip uninstall simple-functions` removes it from `mpm2026` when you no longer need it.)

Let's quickly review the contents of the base folder:

 - **pyproject.toml** - The package's metadata and dependencies (none yet), and its optional `dev` dependencies: `pytest` and `flake8`, the tools we use to check it.
 - **README.md**
 - **LICENSE** - The MIT license (put your name in it).
 - **.gitignore** - Files `git` should not track, such as `__pycache__/` and `*.egg-info/`.
 - **simple_functions/** - Folder containing the 'main' code of this package.
 - **tests/** - Folder containing the tests.

There is no `environment.yml` or `requirements.txt`: as recommended in Lecture 1, `pyproject.toml` is the single source of truth for the package's dependencies.

What's missing is the folder

 - **.github/workflows/** - The folder in which GitHub looks for GitHub Actions workflows. We will add two `.yml` files to it, `flake8.yml` and `pytest-unit-tests.yml`.

### Where do workflows run?

GitHub runs each job of a workflow on a *runner*. GitHub-hosted runners are virtual machines in the cloud: the Linux and Windows ones run in Microsoft Azure's data centres, and the macOS ones in GitHub's own. As stated on their [webpages](https://docs.github.com/en/actions/concepts/runners/github-hosted-runners):

_GitHub offers hosted virtual machines to run workflows. The virtual machine contains an environment of tools, packages, and settings available for GitHub Actions to use._

Each job runs in a fresh virtual machine, which is thrown away when the job finishes. So:

- every job starts from a clean operating system, with Python, `git` and some other common tools preinstalled, and nothing else: it has to install your package and its dependencies itself, every time;
- nothing carries over from your machine, or from the last run, so a passing workflow shows that your code installs and works from scratch;
- you don't look after these machines, and you rent them by the minute: GitHub's standard runners are free on public repositories, and on private ones they use your account's monthly allowance of free minutes (2,000 on a free account; Windows and macOS minutes cost more than Linux ones).

Before we write the workflows, let's take a quick look at our `simple_functions` and the related tests.

Currently, within `simple_functions/functions1.py` is the following code

```python
__all__ = ['my_sum']


def my_sum(iterable):
    """Return the sum of the items in iterable."""
    tot = 0
    for i in iterable:
        tot += i
    return tot
```

that is, just one trivial function. `simple_functions/__init__.py` imports everything listed in `__all__` (`from .functions1 import *`), so that we can write `from simple_functions import my_sum`. In the `tests` folder there is currently just one file, `test_simple_functions.py`, containing the following code:

```python
import pytest

from simple_functions import my_sum


class TestSimpleFunctions:
    """Tests for the functions in functions1.py."""

    @pytest.mark.parametrize('iterable, expected', [
        ([8, 7, 5], 20),
        ((10, -2, 5, -10, 1), 4),
    ])
    def test_my_sum(self, iterable, expected):
        """my_sum adds up the items of lists and tuples."""
        assert my_sum(iterable) == expected
```

Throughout the course of this lecture, we will expand on these functions and tests, and in the process expand our CI through adding further GitHub Actions workflows.

So now, let's add our GitHub Actions workflows and see what they are doing. It is _worth emphasising_ that in order to execute actions workflows for **any** GitHub repository, all that is required is for one or more valid workflows to be placed in the folder `.github/workflows/` (and then pushed to GitHub).

Create the folder `.github/workflows/` in your repository, and in it a file `flake8.yml` containing:

```yml
name: flake8

on:
  # Run the workflow on a push to main, or on a pull request into main
  push:
    branches:
      - main
  pull_request:
    branches:
      - main

jobs:
  flake8:

    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v7

    - name: Set up Python 3.14
      uses: actions/setup-python@v7
      with:
        python-version: "3.14"

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install flake8

    - name: Lint with flake8
      run: |
        flake8 .
```

Let's analyse each segment of the script.

<div class="alert alert-block alert-success">

```yml
name: flake8
```

This is simply the name of the workflow, which is what the `Actions` tab of the repository's GitHub webpage shows (we'll see it shortly)
    
</div>

<div class="alert alert-block alert-success">

```yml
on:
  # Run the workflow on a push to main, or on a pull request into main
  push:
    branches:
      - main
  pull_request:
    branches:
      - main
```
The `on:` segment of the workflow defines when it will be triggered. Here, we're instructing the workflow to trigger when 1) we push to the `main` branch, or 2) a pull request into `main` is opened, or updated by pushing to its branch. A simple `on:` statement could be `on: push` - this would instruct the workflow to execute on any branch when any push to that branch is made. A more complete range of triggers can be found [here](https://docs.github.com/en/actions/reference/workflows-and-actions/events-that-trigger-workflows).

</div>

<div class="alert alert-block alert-success">

```yml
jobs:
  flake8:
```
This is where the 'meat' of the script begins. A workflow run is made up of one or more jobs. Jobs run in parallel by default. For example, a workflow of the form

```yml
jobs:
  job1:
    runs-on: ubuntu-latest
    steps:
    - run: echo "This is job 1"
  job2:
    runs-on: ubuntu-latest
    steps:
    - run: echo "This is job 2"
```
would spawn two jobs, `job1` and `job2`, that are run in parallel (each on its own runner). If we want jobs to run sequentially we can define dependencies via a `needs` tag, e.g.
```yml
jobs:
  job1:
    runs-on: ubuntu-latest
    steps:
    - run: echo "This is job 1"
  job2:
    needs: job1
    runs-on: ubuntu-latest
    steps:
    - run: echo "This is job 2, which starts once job 1 has succeeded"
```
means that `job1` must complete successfully before `job2` is started.
    
In our workflow, we have a single job that we have here called `flake8`.

</div>

<div class="alert alert-block alert-success">

```yml
runs-on: ubuntu-latest
```
This defines the type of machine on which the job will be run: one of GitHub's runners, with Ubuntu Linux, Windows or macOS, or a custom 'self-hosted' machine. Possible options are detailed [here](https://docs.github.com/en/actions/reference/runners/github-hosted-runners). `ubuntu-latest` is currently Ubuntu 24.04; GitHub is moving it to Ubuntu 26.04 between 19 October and 19 November 2026. Name a version (e.g. `ubuntu-24.04`) if you need a job to stay the same.
    
</div>

<div class="alert alert-block alert-success">

Within each of the defined jobs is a series of `steps` that (within a `job`) will be executed sequentially. A new step is started via, e.g., `- uses` or `- name` where the importance of the `-` should be noted (which defines the beginning of a new step). In the absence of `name` being defined for a step some default naming convention will be used.

### Step 1:
    
The first step of our job is
```yml
- uses: actions/checkout@v7
```
This 'uses' is defining a so called 'action' to use (and is where this CI framework derives its name). In GitHub's [own words](https://docs.github.com/en/actions/how-tos/write-workflows/choose-what-workflows-do/find-and-customize-actions):

_"Actions are the building blocks that power your workflow. A workflow can contain actions created by the community, or you can create your own actions directly within your application's repository."_

A large range of community developed actions are freely available at the [GitHub marketplace](https://github.com/marketplace) (which is where all actions used today will come from).

The `checkout` action being used in this first step is simply an action to clone the repository of interest, `simple_functions`, onto the GitHub runner. The `@v7` is the action's version: actions are updated regularly, so check the action's page for the latest major version from time to time.

### Step 2:

```yml
- name: Set up Python 3.14
  uses: actions/setup-python@v7
  with:
    python-version: "3.14"
```
Notice that this step begins with `name`. This simply means that in the actions logs this step will appear with the given name (instead of one derived by default). This step makes use of the `setup-python` action which, as the name suggests, simply sets up the given version of `python` on the runner (here 3.14, as in `mpm2026`; quote the version, since YAML reads an unquoted `3.10` as the number `3.1`). Note that runners will generally have some version of `python` installed by default but this will be the version 'shipped' with the operating system of choice and possibly not the version you wish to use during testing.

### Step 3:

```yml
- name: Install dependencies
  run: |
    python -m pip install --upgrade pip
    pip install flake8
```
Install/upgrade any packages we need. Remember that the runner is a fresh machine: nothing is installed unless a step installs it.
 
### Step 4:

```yml
- name: Lint with flake8
  run: |
    flake8 .
```
Lint with `flake8`!

In the final two steps we're making use of `run`. The [run](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax) command executes the given command-line input in a shell (`bash` on Linux and macOS runners, PowerShell on Windows ones). If a command fails (exits with a non-zero status, as `flake8` does when it finds a problem and `pytest` does when a test fails), the step fails, and so does the job: that is how a workflow turns red.

Single commands can be executed via, e.g.
```yml
    run: pwd
```
Multi-line commands make use of `|`, e.g.
```yml
    run: |
      cd myproject
      python my_sweet_script.py
      echo "my sweet script has run"
```

</div>

Now, in the same folder as `flake8.yml`, create `pytest-unit-tests.yml` containing:

```yml
name: pytest-unit-tests

on:
  # Run the workflow on a push to main, or on a pull request into main
  push:
    branches:
      - main
  pull_request:
    branches:
      - main

jobs:
  pytest:
    name: unit testing
    runs-on: ubuntu-latest

    steps:
    - name: Checkout simple_functions
      uses: actions/checkout@v7

    - name: Set up Python 3.14
      uses: actions/setup-python@v7
      with:
        python-version: "3.14"

    - name: Install the package and its dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -e ".[dev]"

    - name: Test with pytest
      run: |
        pytest tests/
```

You'll see that most of the content is very similar to that of the `.yml` we have just studied. Some new additions to point out are the following:

<div class="alert alert-block alert-success">

```yml
    - name: Install the package and its dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -e ".[dev]"
```
This step installs our package, with the dependencies declared in its `pyproject.toml`, including the optional `dev` ones (`pytest` and `flake8`): exactly what we did on our own machine.

</div>

<div class="alert alert-block alert-success">

```yml
    - name: Test with pytest
      run: |
        pytest tests/
```
The final step in the job is executing `pytest`. If our tests pass we'll get a green light - if one fails we'll get a red light! This interacts with the PR interface we saw last week (Lecture 1): the result of each workflow is shown on the pull request, next to the merge button. By default GitHub still lets you merge a PR whose checks failed (with a warning); to stop that, the repository's owner adds a branch protection rule (`Settings` → `Branches`) that requires the checks to pass before merging. (Protected branches are available for public repositories, and for private ones on paid plans.)
        
</div>

Now commit the two workflows and push them:

```bash
git add .github
git commit -m "Add flake8 and pytest workflows"
git push
```

On the webpage of your repository, click on the `Actions` tab. You'll see something like the following (this screenshot is from last year's version of the repository):

<img src="figures/mpm_actions.png" width=1000 height=1000 />

This tab provides us with a summary of which workflows are being executed (here `flake8` and `pytest-unit-tests`), when they were last executed and whether they produced a pass (green tick) or a fail (red cross). Click on a run, then on a job, to see the log of every step.

**Try it:** on a new branch, break `my_sum` (e.g. `tot += i + 1`), push the branch and open a pull request. Watch the checks run on the pull request, and fail. Then close the pull request without merging it.

## Adding more 'simple functions': how does `numpy` compute $\pi$, `sin(x)` etc.?

Software users often make use of basic functions such as $\pi$, `sin`, `cos` and `tan` etc. without thinking about the underlying algorithm used to actually produce the result.

An important aspect of this course is that you start thinking about algorithms at a more fundamental level and that **you're** able to develop and produce such algorithms (and not just be a user).

Let's consider how we would write a function to compute $\pi$. [Srinivasa Ramanujan](https://en.wikipedia.org/wiki/Srinivasa_Ramanujan) produced many rapidly convergent series for approximating $\pi$. One of them is as follows:
\begin{equation}
 \frac{1}{\pi}=\frac{2\sqrt{2}}{9801}\sum_{k=0}^{\infty}\frac{(4k)!(1103+26390k)}{k!^4(396^{4k})}.
\end{equation}
Each term adds about eight correct digits. Note: record computations of $\pi$ today (314 trillion digits in 2025) use a formula published by the Chudnovsky brothers in 1988, which builds on Ramanujan's work.

First, in order to implement this formula we'll need to write a `factorial` function. (We could also implement a square root function, but for that we'll just use `numpy.sqrt` for now). In `simple_functions/functions1.py` let's add:

```python
@cache
def factorial(n):
    """Return n! (n factorial) for a non-negative integer n."""
    return n * factorial(n-1) if n else 1
```
`cache` should be imported from `functools` (`from functools import cache`, at the top of the file), and let's remember to add `'factorial'` to `__all__`.

Then, let's add a test for this function to the `TestSimpleFunctions` class in `tests/test_simple_functions.py` (remember to import `factorial`!), e.g.

```python
    @pytest.mark.parametrize('number, expected', [
        (5, 120),
        (3, 6),
        (1, 1),
        (0, 1),
    ])
    def test_factorial(self, number, expected):
        """factorial is right for small numbers, including 0! = 1."""
        assert factorial(number) == expected
```
After ensuring our `factorial` function is working, let's move on to our computation of $\pi$.

Let's add a file called `simple_functions/constants.py` and implement Ramanujan's equation. Our new file should contain the following:

```python
from functools import cache

from numpy import sqrt

from simple_functions.functions1 import factorial

__all__ = ['pi']


def pi(terms=1):
    """Approximate pi by summing Ramanujan's series from k = 0 to terms."""
    return 1./(2.*sqrt(2.)/9801.*rsum(terms))


@cache
def rsum(n):
    """Return the sum of the terms k = 0, 1, ..., n of Ramanujan's series."""
    t = factorial(4*n)*(1103+26390*n)/(factorial(n)**4*396**(4*n))
    return t + rsum(n-1) if n else t
```

For `pi` to be importable from the package itself (`from simple_functions import pi`), `simple_functions/__init__.py` must import it, as it does `my_sum`. Add this line to `__init__.py`:

```python
from .constants import *  # noqa
```

Then, let's add a new test file called `tests/test_constants.py`. This should look something like the following (`pytest.approx` compares floats with a tolerance, as in Lecture 4; given only `abs`, the tolerance is absolute):

```python
import numpy as np
import pytest

from simple_functions import pi


class TestPi:
    """Tests for the constants in constants.py."""

    def test_pi(self):
        """Three terms of Ramanujan's series give pi to double precision."""
        assert pi(2) == pytest.approx(np.pi, abs=1e-12)
```

Check that the tests pass (`pytest` should now report `7 passed`) and that `flake8 .` is still happy. With that done, let's commit, push and check that our workflows are correctly executing these tests!

### Passing on my machine ≠ passing

The `flake8` workflow passes, but `pytest-unit-tests` fails. Its log (click on the failed run, then on the `unit testing` job, then on the `Test with pytest` step) shows

```text
simple_functions/constants.py:3: in <module>
    from numpy import sqrt
E   ModuleNotFoundError: No module named 'numpy'
=========================== short test summary info ============================
ERROR tests/test_constants.py
ERROR tests/test_simple_functions.py
!!!!!!!!!!!!!!!!!!! Interrupted: 2 errors during collection !!!!!!!!!!!!!!!!!!!!
```

The tests pass on your machine because `mpm2026` has NumPy. The runner is a fresh virtual machine: it has only what the workflow installs, and `pip install -e ".[dev]"` installs only the dependencies that `pyproject.toml` declares. We never declared NumPy. (Both test files fail, not just `test_constants.py`: importing `simple_functions` now imports `constants.py`, and with it NumPy.) You can see the same failure on your own machine by installing the package in a new, empty `venv`, as in Lecture 1.

The fix belongs in the package, not in the workflow. In `pyproject.toml`, change

```toml
dependencies = []
```
to
```toml
dependencies = ["numpy"]
```
then commit and push again. This time both workflows pass, and anyone who installs your package with `pip` gets NumPy with it.

*Takeaway:* CI checks that your package installs and works on a clean machine with only what it declares, which is all your users will have.

### Exercise:

- Add a new function to compute `sin(x)` to `functions1.py` (or to a new, e.g., `trig_functions.py` file, imported in `__init__.py` like `constants.py`)
- Add a related parameterised (i.e. test a few different inputs automatically) test to `test_simple_functions.py`
- Ensure the test is passing locally
- Finally, commit the changes and push them to GitHub and check that they are now indeed being executed by the pytest-unit-tests.yml workflow

#### _Notes:_
- How trigonometric functions are actually computed by software such as numpy is actually quite interesting and not entirely trivial (see [this](https://stackoverflow.com/questions/2284860/how-does-c-compute-sin-and-other-math-functions) discussion for example). For this basic exercise you can assume that the input is real and between 0 and $2\pi$ radians.
- A quick and easy implementation is via a Taylor series expansion. The expansion can be found within [this webpage](https://en.wikipedia.org/wiki/Sine).
- You should test your result against, e.g., numpy's result. Compare floats with a tolerance, as in Lecture 4: `assert my_sin(x) == pytest.approx(np.sin(x))`, or [`np.testing.assert_allclose`](https://numpy.org/doc/stable/reference/generated/numpy.testing.assert_allclose.html) for arrays. Near zero (e.g. at $x = \pi$) a relative tolerance means little, so give an absolute one too (`pytest.approx` has one by default, $10^{-12}$; `assert_allclose` needs `atol=...`).
- Think about when this test is not good and how to improve the algorithm e.g. what if the input is $>2\pi$? Does it work if the input is complex?

## Some other useful GitHub Actions features

A short reference: you'll meet these in other projects' workflows. The [workflow syntax reference](https://docs.github.com/en/actions/reference/workflows-and-actions/workflow-syntax) has the details.

#### [Matrices](https://docs.github.com/en/actions/how-tos/write-workflows/choose-what-workflows-do/run-job-variations)

Run the same job on several operating systems and/or with several versions of Python, in parallel:

```yml
jobs:
  pytest:
    name: ${{ matrix.os }}, Python ${{ matrix.python-version }}
    runs-on: ${{ matrix.os }}

    strategy:
      # Let every job in the matrix finish, even if one of them fails
      fail-fast: false
      matrix:
        os: [ubuntu-latest, windows-latest, macos-latest]
        python-version: ["3.12", "3.13", "3.14"]

    steps:
    - uses: actions/checkout@v7

    - name: Set up Python ${{ matrix.python-version }}
      uses: actions/setup-python@v7
      with:
        python-version: ${{ matrix.python-version }}

    # ... then install and test, as in pytest-unit-tests.yml
```
This makes nine jobs, one for each combination. Quote the Python versions: unquoted, YAML reads `3.10` as the number `3.1`. By default (`fail-fast: true`), as soon as one job in the matrix fails, GitHub cancels the others; `fail-fast: false` lets them all finish, so that you see every combination that fails. (`include` and `exclude` add or remove particular combinations.)

#### Scheduled runs (cron)

It is often useful to have workflows trigger on a [schedule](https://docs.github.com/en/actions/reference/workflows-and-actions/events-that-trigger-workflows#schedule), e.g. to catch a new release of a dependency that breaks your code. Such a schedule can be defined using [POSIX cron syntax](https://pubs.opengroup.org/onlinepubs/9699919799/utilities/crontab.html#tag_20_25_07):

```yml
on:
  schedule:
    - cron: '0 3 11,25 * *'
```
This states that the workflow should be executed at 0 minutes, 3 hours (UTC) on the 11th and 25th day of every month. Scheduled workflows run on the default branch only.

#### workflow_dispatch

Sometimes it's handy to have a 'push button' to run a workflow. Adding [workflow_dispatch](https://docs.github.com/en/actions/how-tos/manage-workflow-runs/manually-run-a-workflow) to its triggers, e.g.

```yml
on:
  push:
    branches:
      - main
  workflow_dispatch:
```
introduces a 'Run workflow' button on the Actions tab under the relevant workflow (once the workflow is on the default branch), as shown in the (older) screenshot below

<img src="figures/actions2.png" width=1000 height=1000 />

#### `if` conditions

A step with an `if:` runs only when its condition is true:

```yml
    - name: A step that only runs on Windows
      if: runner.os == 'Windows'
      run: echo "Hello from Windows"
```
`runner.os` is one of the [contexts](https://docs.github.com/en/actions/reference/workflows-and-actions/contexts) (information about the run) that conditions can use. A job can also have an `if:`; e.g. `if: ${{ always() }}` lets a job with `needs:` run even when a job it needs has failed.

#### Step outputs

A value from one step can be used in a later step by writing `name=value` to the file whose path is in `$GITHUB_OUTPUT`, and giving the step an `id`:

```yml
    - name: Make some value
      id: msv
      run: echo "some_output=some_value" >> "$GITHUB_OUTPUT"

    - name: Use some value
      run: echo "The value was ${{ steps.msv.outputs.some_output }}"
```

#### Self-hosted runners

Instead of running on the default runners provided, you can also set jobs to be run on [self-hosted](https://docs.github.com/en/actions/concepts/runners/self-hosted-runners) runners. These could be, e.g., a cloud virtual machine or a machine of your own (with a GPU, say). Once the runner is configured, jobs can be sent to this machine via:

```yml
runs-on: self-hosted
```
If you have several self-hosted runners, custom labels send different jobs to different runners, e.g.

```yml
runs-on: [self-hosted, gpu]
```

#### And many more...

We've just scratched the surface of what can be achieved with GitHub Actions. As you delve into more advanced development projects maybe you'll need to start exploring some of the more creative ways to make use of actions!

## GitHub `Secrets`

Sometimes a workflow requires sensitive information (e.g. a password or an API token). Clearly, you don't want such information displayed in your repository, but your workflows still need to use it. In such situations GitHub Secrets come in handy.

To add a secret, open the repository's `Settings` tab, then `Secrets and variables` → `Actions`, and click `New repository secret`. A workflow can then use it via `${{ secrets.MY_API_TOKEN }}`, e.g. by putting it in an environment variable for one step:

```yml
    - name: Upload the results
      env:
        API_TOKEN: ${{ secrets.MY_API_TOKEN }}
      run: python upload_results.py   # reads the token from $API_TOKEN
```
GitHub replaces secrets with `***` if they appear in a log, and does not give them to workflows triggered by pull requests from forks. Every workflow also gets a temporary token of its own, `secrets.GITHUB_TOKEN`, for working with its repository.

## Some useful actions

We've already seen `actions/checkout` and `actions/setup-python` in use. Below is an _extremely_ non-exhaustive list of some other useful actions (with their current major versions, October 2026):

#### setup-miniconda

[`conda-incubator/setup-miniconda`](https://github.com/conda-incubator/setup-miniconda) creates a conda environment, e.g. from an `environment.yml` file (Lecture 1), instead of `setup-python`:

```yml
    - uses: conda-incubator/setup-miniconda@v4
      with:
        miniforge-version: latest
        environment-file: environment.yml
        activate-environment: mpm2026

    - name: Test with pytest
      shell: bash -el {0}   # a login shell, so that the environment is activated
      run: |
        pytest tests/
```

#### upload-artifact _and_ download-artifact

_"Use [artifacts](https://docs.github.com/en/actions/tutorials/store-and-share-data) to share data between jobs in a workflow and store data once that workflow has completed."_

This is incredibly useful when you want to collect and check, e.g., a set of results files from runs on various runners. Here are some example snippets of an upload (in each job of a matrix) and then a download (in a later job):

-----------------------------------------------------------------------------------
```yml
    - name: Upload the results
      uses: actions/upload-artifact@v7
      with:
        name: results-${{ matrix.os }}
        path: results/
        if-no-files-found: error         # optional, but helpful
        retention-days: 7                # optional
```
-----------------------------------------------------------------------------------
```yml
    - name: Download the results of every job
      uses: actions/download-artifact@v8
      with:
        path: results
        merge-multiple: true             # put the files from all the artifacts in one folder
```
-----------------------------------------------------------------------------------

#### Codecov

Lecture 4 measured test coverage with `pytest-cov`. [Codecov](https://about.codecov.io/) collects the coverage reports from your CI runs, shows how coverage changes over time and comments on pull requests. After signing in to Codecov with your GitHub account, a job like this uploads the report:

```yml
jobs:
  coverage:
    runs-on: ubuntu-latest
    permissions:
      contents: read    # to check out the repository
      id-token: write   # lets Codecov check which repository the upload comes from

    steps:
    - uses: actions/checkout@v7

    - uses: actions/setup-python@v7
      with:
        python-version: "3.14"

    - name: Install the package, with pytest-cov
      run: |
        python -m pip install --upgrade pip
        pip install -e ".[dev]" pytest-cov

    - name: Test, measuring coverage
      run: |
        pytest --cov=simple_functions --cov-report=xml tests/

    - name: Upload coverage to Codecov
      uses: codecov/codecov-action@v7
      with:
        use_oidc: true
```

#### pypi-publish

[`pypa/gh-action-pypi-publish`](https://github.com/pypa/gh-action-pypi-publish) uploads a package to PyPI (or TestPyPI), usually when you publish a release, using PyPI's *trusted publishing* (see the end of the first half): no API token needs to be stored anywhere.

### Exercise:

- Add a new function of your choice that requires a new dependency (e.g. a function that uses SciPy).
- Add some appropriate tests for this new function. Before you push: will they pass on the runner?
- Create a new, or expand an old, workflow and utilise matrices to run all tests on `ubuntu-24.04`, `ubuntu-26.04`, `macos-latest` and (if you like) `windows-latest`. Notice how the use of matrices parallelises the jobs! **TIP**: Only add one matrix entry at a time!
- Finally, do the same for the package you built in Exercise 2 of the first half (`python-packaging.ipynb`): give it a GitHub repository of its own, with a `flake8` workflow and a `pytest` workflow.

**Note:** Worked solutions to this lecture's exercises will be released after the lecture.

## Self-check quiz

Today's lecture has a short multiple-choice [self-check quiz](quiz.ipynb) on its main points, covering both halves: `python-packaging.ipynb` and this notebook. Open the notebook and click an option under each question. Take the quiz once you have worked through both halves, and before you start any part of the coursework that relies on them. The answers, with explanations, will be released later so that you can check your own. The quiz is not assessed, but it is good preparation for the multiple-choice part of the consolidation test.
